# [제출물 ②] 나의 SDD 확장 기록 — 강원생활도우미앱

이름/학번: 이규환 / ______    |    내가 고른 확장 기능: **추천 이유 문장 출력 (중간)**

이 노트북은 **채워서 제출**합니다. 레거시 확장 6단계를 따라가며, 각 칸을 직접 채우세요.
검증은 화면이 아니라 **로직 함수**에 겁니다(그래서 노트북에서 끝까지 돌아갑니다).

**① 이해 → ② 현재 동작 고정 → ③ 확장 명세 → ④ 생성 → ⑤ 검증(신규+회귀) → ⑥ 통합·문서화**

---
## 0. 준비 — 데이터 & 기존 로직

엑셀이 없어도 노트북이 돌아가게, 내 앱 데이터를 본뜬 샘플을 만들고 기존 `join_data`를 가져옵니다.

In [1]:
import pandas as pd

# 내 앱의 두 시트(장소정보/추천정보)를 본뜬 샘플
def make_sample():
    place = pd.DataFrame([
        ['P01','안목해변카페','강릉','카페',9000,4.6,'아니오'],
        ['P02','강릉책방카페','강릉','카페',5000,4.2,'아니오'],
        ['P03','바다뷰카페','강릉','카페',8000,4.8,'예'],
    ], columns=['place_id','이름','지역','유형','예산','평점','예약필요'])
    recommend = pd.DataFrame([
        ['P01','휴식','맑은날','친구'],
        ['P02','휴식','맑은날','친구'],
        ['P03','휴식','맑은날','친구'],
    ], columns=['place_id','추천목적','추천상황','추천대상'])
    return place, recommend

def join_data(place_df, recommend_df):
    return pd.merge(recommend_df, place_df, on='place_id', how='left')

place_df, recommend_df = make_sample()
merged_df = join_data(place_df, recommend_df)
merged_df

,place_id,추천목적,추천상황,추천대상,이름,지역,유형,예산,평점,예약필요
0,P01,휴식,맑은날,친구,안목해변카페,강릉,카페,9000,4.6,아니오
1,P02,휴식,맑은날,친구,강릉책방카페,강릉,카페,5000,4.2,아니오
2,P03,휴식,맑은날,친구,바다뷰카페,강릉,카페,8000,4.8,예


---
## 1. 이해 — 기존 코드가 무엇을 하나 (직접 작성)

**내가 쓴 프롬프트:**
```
다음은 제 강원생활도우미앱의 검색 부분입니다. [filter_recommendations 코드 붙여넣음]
무슨 일을 하는지 요약하고, 결과로 어떤 열의 DataFrame을 만드는지 알려주세요.
```

**AI 답을 검증한 결과 정리:**
- 입력: 조인된 DataFrame(merged_df) + 검색 조건 5개(지역, 추천목적, 추천상황, 추천대상, 예산 상한)
- 출력: 5개 조건을 모두 만족하는 행만 남긴 DataFrame. 열은 원본 그대로(place_id, 추천목적, 추천상황, 추천대상, 이름, 지역, 유형, 예산, 평점, 예약필요)
- AI 설명 중 틀렸거나 보완한 점: AI가 처음에 "예산이 정확히 같은 행"이라고 설명했는데, 실제 코드는 `<=`라서 **예산 이하**가 맞음. 직접 코드를 보고 정정함. 또 필터만 하고 원본은 안 바꾼다는 것(뷰 반환)을 실행해서 확인함.

In [2]:
# 확장과 닿는 기존 '로직'을 함수로 떼어내 두기 (UI 없이)
def filter_recommendations(df, 지역, 추천목적, 추천상황, 추천대상, 예산):
    return df[
        (df['지역'] == 지역) & (df['추천목적'] == 추천목적) &
        (df['추천상황'] == 추천상황) & (df['추천대상'] == 추천대상) &
        (df['예산'] <= 예산)
    ]

filter_recommendations(merged_df, '강릉', '휴식', '맑은날', '친구', 10000)

,place_id,추천목적,추천상황,추천대상,이름,지역,유형,예산,평점,예약필요
0,P01,휴식,맑은날,친구,안목해변카페,강릉,카페,9000,4.6,아니오
1,P02,휴식,맑은날,친구,강릉책방카페,강릉,카페,5000,4.2,아니오
2,P03,휴식,맑은날,친구,바다뷰카페,강릉,카페,8000,4.8,예


---
## 2. 현재 동작 고정 — 회귀 기준 (직접 작성)

확장 전, 지금 잘 되는 동작을 '박제'합니다. 이 값이 바뀌면 = 기존을 깨뜨린 것.

내가 지킬 기존 동작: **강릉/휴식/맑은날/친구/예산 10000원 검색 결과의 개수와 장소 목록**

In [3]:
base = filter_recommendations(merged_df, '강릉', '휴식', '맑은날', '친구', 10000)
기준_개수 = len(base)
기준_이름들 = sorted(base['이름'].tolist())
print('회귀 기준 개수:', 기준_개수)
print('회귀 기준 장소:', 기준_이름들)

회귀 기준 개수: 3
회귀 기준 장소: ['강릉책방카페', '바다뷰카페', '안목해변카페']


---
## 3. 확장 명세 + 접점 (직접 작성)

### 확장 기능 명세
- 무엇을: 검색 결과의 각 장소에 대해 **왜 추천하는지 설명하는 한국어 문장**을 만들어 '추천이유' 열로 붙인다
- 입력: 검색 결과 DataFrame (이름, 예산, 평점, 예약필요, 추천목적 열을 포함)
- 출력: 입력을 **복사한 뒤** '추천이유' 열이 추가된 새 DataFrame (함수 형태: `make_reasons(df)`)
- 제약: 기존 함수·열은 바꾸지 않는다. 입력은 복사해서 사용(원본 불변). 예약필요가 '아니오'인 곳만 "예약 없이 바로 방문 가능" 문구를 넣는다
- 완료 조건 (숫자로 확인 가능):
  1. 답을 미리 아는 1행 입력에서 문장이 예측 문자열과 **정확히 일치** (True)
  2. 출력 행 개수 == 입력 행 개수
  3. 출력에 '추천이유' 열이 존재하고, 원본에는 생기지 않음

### 접점(인터페이스) 명세
- 기존 → 신규로 무엇을 넘기나: `filter_recommendations`의 반환 DataFrame을 그대로 넘긴다
- 신규가 의존하는 열: 이름, 예산, 평점, 예약필요, 추천목적 (5개)

---
## 4. 생성 — AI로 새 코드 받기 (직접 작성)

**내가 쓴 프롬프트:**
```
다음 명세대로 새 함수를 만들어줘. 기존 함수·코드는 절대 수정하지 말고, 새 함수만 추가해줘.
- 무엇을: 검색 결과 각 장소에 추천 이유 문장을 만들어 '추천이유' 열로 추가
- 입력: 이름, 예산, 평점, 예약필요, 추천목적 열이 있는 DataFrame
- 출력: '추천이유' 열이 추가된 새 DataFrame (함수 형태: make_reasons(df))
- 제약: 입력은 복사해서 사용(원본 불변). 예약필요가 '아니오'일 때만 "예약 없이 바로 방문 가능" 추가
- 완료 조건: "OO은(는) 평점 X점, 예산 Y원(, 예약 없이 바로 방문 가능)이라서 Z하기 좋은 곳입니다." 형식
고등학생이 이해할 수 있는 코드로, 설명은 짧게.
```

In [4]:
# AI가 준 새 함수 코드 (기존 함수는 그대로!)
def make_reasons(df):
    result = df.copy()                       # 원본 불변: 복사해서 사용
    reasons = []
    for _, row in result.iterrows():         # 한 행(장소)씩 문장 조립
        parts = [f"{row['이름']}은(는) 평점 {row['평점']}점"]
        parts.append(f"예산 {int(row['예산'])}원")
        if row['예약필요'] == '아니오':       # 예약 불필요한 곳만 문구 추가
            parts.append('예약 없이 바로 방문 가능')
        sentence = ', '.join(parts) + f"이라서 {row['추천목적']}하기 좋은 곳입니다."
        reasons.append(sentence)
    result['추천이유'] = reasons              # 새 열로 붙임
    return result

# 일단 실행해 보기 (이건 '실행'일 뿐 '판정'은 5단계에서)
make_reasons(filter_recommendations(merged_df, '강릉', '휴식', '맑은날', '친구', 10000))[['이름','추천이유']]

,이름,추천이유
0,안목해변카페,"안목해변카페은(는) 평점 4.6점, 예산 9000원, 예약 없이 바로 방문 가능이라..."
1,강릉책방카페,"강릉책방카페은(는) 평점 4.2점, 예산 5000원, 예약 없이 바로 방문 가능이라..."
2,바다뷰카페,"바다뷰카페은(는) 평점 4.8점, 예산 8000원이라서 휴식하기 좋은 곳입니다."


---
## 5. 검증 — 신규 + 회귀 (직접 작성)

**신규 검증**은 '답을 미리 아는 작은 입력(테스트 데이터)'을 만들어 예측과 대조합니다.
**회귀 검증**은 2단계에서 박제한 기존 동작이 그대로인지 확인합니다.

> 검증 기준과 테스트는 내가 만들었음. 테스트 데이터는 1행짜리라서 문장 전체를 손으로 미리 써서 예측할 수 있음.

In [5]:
print('=== 신규 검증 (테스트 데이터 + 드라이버) ===')
# 테스트 데이터: 1행이라 결과 문장을 미리 손으로 쓸 수 있음(답을 미리 앎)
mini = pd.DataFrame({
    '이름': ['테스트카페'],
    '예산': [3000],
    '평점': [4.5],
    '예약필요': ['아니오'],
    '추천목적': ['휴식'],
})
# 테스트 드라이버: 넣고 호출해 예측과 대조
r = make_reasons(mini)
expected = '테스트카페은(는) 평점 4.5점, 예산 3000원, 예약 없이 바로 방문 가능이라서 휴식하기 좋은 곳입니다.'
print('문장이 예측과 일치      :', r.iloc[0]['추천이유'] == expected)
print('행 개수 그대로          :', len(r) == len(mini))
print('추천이유 열 추가됨      :', '추천이유' in r.columns)
print('원본 mini 불변          :', '추천이유' not in mini.columns)

print('=== 회귀 검증 (기존 동작 그대로인가) ===')
재검색 = filter_recommendations(merged_df, '강릉', '휴식', '맑은날', '친구', 10000)
print('기존 검색 개수 그대로   :', len(재검색) == 기준_개수)
print('기존 검색 장소 그대로   :', sorted(재검색['이름'].tolist()) == 기준_이름들)
print('원본에 추천이유 열 없음 :', '추천이유' not in merged_df.columns)

=== 신규 검증 (테스트 데이터 + 드라이버) ===
문장이 예측과 일치      : True
행 개수 그대로          : True
추천이유 열 추가됨      : True
원본 mini 불변          : True
=== 회귀 검증 (기존 동작 그대로인가) ===
기존 검색 개수 그대로   : True
기존 검색 장소 그대로   : True
원본에 추천이유 열 없음 : True


---
## 6. 통합 & 문서화 (직접 작성)

app.py에서 기존 검색 결과 출력 뒤에 새 함수 호출만 더했습니다. (기존 코드는 그대로!)

```python
# search_recommendations 안, 기존 결과 출력 뒤에
if len(result) > 0:
    st.dataframe(result)
    if st.checkbox('추천 이유 보기'):
        for s in make_reasons(result)['추천이유']:
            st.info(s)
else:
    st.warning('조건에 맞는 추천 장소가 없습니다.')
```

**더한 것(새 함수·새 메뉴):** `make_reasons` 함수 1개 + '추천 이유 보기' 체크박스

**건드리지 않은 것(보호한 기존 함수):** `load_data`, `join_data`, `filter_recommendations`(검색 필터), 차트 기능 전부

**회귀 검증 결과(기존 기능 그대로?):** 기존 검색 개수·장소 그대로, 원본 DataFrame에 새 열 안 생김 — 전부 통과

**AI를 어떻게 썼나(프롬프트→검증→수정):** 1단계에서 기존 코드 설명을 시켰고, 예산 조건을 '같음'이라고 잘못 설명한 부분을 코드를 직접 읽고 정정함. 4단계에서 명세 5요소를 그대로 프롬프트로 옮겨 새 함수만 받았고, 5단계 테스트 데이터·기준은 AI에게 맡기지 않고 내가 만들어 검증함.

**접점/검증에서 배운 점:** 새 함수가 검색 결과를 `copy()`해서 쓰니 원본이 안전했다. 접점 명세에서 의존하는 열 5개를 미리 적어 두니, 테스트 데이터를 그 5개 열만으로 최소하게 만들 수 있었다.

**(선택) 앱 실행 화면 스크린샷 / 짧은 시연 링크:** (앱 실행 후 스크린샷 첨부)

---
### 제출 전 셀프 체크
- [x] 모든 코드 셀이 위에서 아래로 오류 없이 실행된다
- [x] 신규 검증(답 아는 입력)과 회귀 검증이 모두 통과한다
- [x] 기존 함수를 바꾸지 않고 새 함수로 확장했다
- [x] AI가 준 코드를 한 줄씩 설명할 수 있다